# AksaraLine 1/2 — build the glyph pool and render the corpus

CPU only. Writes everything to `/kaggle/working/build`, which Kaggle saves as
this kernel's output; the training kernel then attaches that output instead of
re-rendering. Splitting the two matters because a GPU session is capped at 9h
and `/kaggle/working` does not survive between runs — folding rendering into
the training kernel would repeat ~1h of CPU work every time and risk losing the
whole session to a timeout.


In [ ]:
import os, sys, subprocess, zipfile, time
from pathlib import Path

BRANCH  = 'aksara-seq'
REPO    = Path('/kaggle/working/aksara_OCR')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/phoenixfin/aksantara-ocr.git',
                    str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'aksara_seq' / 'src'))
print('commit:', subprocess.run(['git','rev-parse','--short','HEAD'],
      capture_output=True, text=True).stdout.strip())


In [ ]:
DATA_ROOT = Path('/kaggle/working/data/clean')
SCRIPTS   = ['Sunda', 'Jawa', 'Bali', 'Lontara']
INPUT     = Path('/kaggle/input')

# Do not assume where Kaggle mounts a dataset. It has used both
# /kaggle/input/<slug> and /kaggle/input/datasets/<owner>/<slug>, so locate
# each script by searching a few levels down instead of hardcoding a path.
def _dirs(root, depth):
    out = [root]
    frontier = [root]
    for _ in range(depth):
        nxt = []
        for d in frontier:
            try:
                nxt += [q for q in d.iterdir() if q.is_dir()]
            except OSError:
                pass
        out += nxt
        frontier = nxt
    return out

CANDIDATES = _dirs(INPUT, 4) if INPUT.is_dir() else []
print(f'searching {len(CANDIDATES)} mounted directories')

def locate(script):
    for d in CANDIDATES:
        if d.name == script and any(d.iterdir()):
            return 'dir', d
    for ext in ('.bin', '.zip'):
        for d in CANDIDATES:
            f = d / f'{script}{ext}'
            if f.is_file():
                return 'archive', f
    return None, None

DATA_ROOT.mkdir(parents=True, exist_ok=True)
for s in SCRIPTS:
    if (DATA_ROOT / s).is_dir() or (DATA_ROOT / s).is_symlink():
        continue
    kind, found = locate(s)
    assert kind, (f'no source for {s}; looked for a {s}/ directory or '
                  f'{s}.bin/.zip under {INPUT}, saw '
                  f'{sorted(q.name for q in CANDIDATES)[:20]}')
    if kind == 'dir':
        os.symlink(found, DATA_ROOT / s)
        print(f'{s}: linked {found}')
    else:
        t0 = time.time()
        with zipfile.ZipFile(found) as z:
            z.extractall(DATA_ROOT)
        print(f'{s}: extracted {found.name} in {time.time()-t0:.0f}s')

for s in SCRIPTS:
    n = sum(1 for q in (DATA_ROOT / s).rglob('*') if q.is_file())
    print(f'  {s:9s} {n} files')
    assert n > 1000, f'{s} looks incomplete ({n} files)'


## Glyph pool

Verifies each (onset × vowel) grid against its known class count before writing.


In [ ]:
BUILD  = Path('/kaggle/working/build')
GLYPHS = BUILD / 'glyphs'
!python aksara_seq/scripts/01_build_glyph_pool.py     --data-root {DATA_ROOT} --out {GLYPHS} --workers 4 --no-hash


## Render, then verify

The verifier re-derives split disjointness from the written labels rather than
trusting the generator, and checks every bounding box.


In [ ]:
CORPUS = BUILD / 'corpus' / 'v1'
!python aksara_seq/scripts/02_render_corpus.py     --config aksara_seq/configs/corpus_v1.yaml     --glyph-cache {GLYPHS} --out {CORPUS}
!python aksara_seq/scripts/03_verify_corpus.py --corpus {CORPUS}


## Trim the output

The glyph cache is ~500 MB of intermediate that the trainer never reads —
dropping it keeps this kernel's output small enough to attach comfortably.


In [ ]:
import shutil, zipfile

# Emit a handful of archives, not 100,000 loose files. A previous run
# completed and reported 1.76 GB of output, yet the attached output mounted
# empty in the training kernel -- the same failure mode as Kaggle silently
# dropping a dataset built from a many-file zip. Archives sidestep it, and
# ZIP_STORED keeps it fast since the PNGs are already compressed.
for s in SCRIPTS:
    src = CORPUS / s
    out = BUILD / f'corpus_{s}.zip'
    n = 0
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_STORED) as z:
        for p in sorted(src.rglob('*')):
            if p.is_file():
                z.write(p, arcname=str(Path(s) / p.relative_to(src)))
                n += 1
    print(f'{s}: {n} files -> {out.stat().st_size/1e6:.0f} MB')
    shutil.rmtree(src)

shutil.copy2(CORPUS / 'dataset_meta.json', BUILD / 'dataset_meta.json')
shutil.rmtree(CORPUS, ignore_errors=True)
shutil.rmtree(GLYPHS, ignore_errors=True)
shutil.rmtree('/kaggle/working/data', ignore_errors=True)
shutil.rmtree(REPO, ignore_errors=True)

files = sorted(p for p in BUILD.rglob('*') if p.is_file())
total = sum(p.stat().st_size for p in files)
print(f'
output: {len(files)} files, {total/1e9:.2f} GB')
for p in files:
    print(f'  {p.relative_to(BUILD)}  {p.stat().st_size/1e6:.0f} MB')
